# Ch04 -- 機率分佈：離散與連續

> **預估時間：** 2 小時  
> **前置章節：** [Ch03 -- 機率基礎與條件機率](03_機率基礎與條件機率.ipynb)  
> **資料集：** `datasets/titanic_train.csv`

## 學習目標

1. 理解隨機變數 (Random Variable) 的分類：離散 vs 連續
2. 掌握 PMF、PDF、CDF 三種函數的定義與圖形化
3. 認識常見離散分佈：Bernoulli、Binomial、Poisson
4. 認識常見連續分佈：Uniform、Normal、t、Chi-squared、F
5. 學會使用 QQ Plot 判斷資料是否服從常態分佈 (Normal Distribution)
6. 實務上如何透過對數轉換 (Log Transformation) 將偏態資料近似常態化

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# 中文字型設定（macOS）
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'Heiti TC']
plt.rcParams['axes.unicode_minus'] = False

# 圖表風格
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

np.random.seed(42)

In [ ]:
# 載入 Titanic 資料集
df = pd.read_csv('datasets/titanic_train.csv')
df.head()

---
## Part A -- 隨機變數與分佈

### A1. 隨機變數 (Random Variable)

隨機變數是一個將隨機實驗的結果對應到數值的函數。依照取值方式分為兩類：

| 類型 | 定義 | 範例 |
|------|------|------|
| **離散隨機變數 (Discrete)** | 取值為可數的有限或無限集合 | 擲骰子的點數、乘客存活人數 |
| **連續隨機變數 (Continuous)** | 取值為某區間內任一實數 | 身高、體重、票價 (Fare) |

### A2. 機率質量函數 PMF vs 機率密度函數 PDF

- **PMF (Probability Mass Function)**：描述離散隨機變數在各取值的機率，$P(X = x)$
- **PDF (Probability Density Function)**：描述連續隨機變數的機率密度，$f(x)$。注意：$P(X = x) = 0$，機率由面積（積分）決定
- **CDF (Cumulative Distribution Function)**：$F(x) = P(X \leq x)$，適用於離散與連續

In [ ]:
# 示範：離散 PMF vs 連續 PDF vs CDF
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- 離散：Binomial PMF ---
n, p = 10, 0.3
x_binom = np.arange(0, n + 1)
pmf_vals = stats.binom.pmf(x_binom, n, p)
axes[0, 0].bar(x_binom, pmf_vals, color='steelblue', edgecolor='white')
axes[0, 0].set_title('離散：Binomial PMF (n=10, p=0.3)', fontsize=13)
axes[0, 0].set_xlabel('k')
axes[0, 0].set_ylabel('P(X = k)')

# --- 離散：Binomial CDF ---
cdf_binom = stats.binom.cdf(x_binom, n, p)
axes[0, 1].step(x_binom, cdf_binom, where='mid', color='steelblue', linewidth=2)
axes[0, 1].scatter(x_binom, cdf_binom, color='steelblue', zorder=5)
axes[0, 1].set_title('離散：Binomial CDF (n=10, p=0.3)', fontsize=13)
axes[0, 1].set_xlabel('k')
axes[0, 1].set_ylabel('P(X <= k)')

# --- 連續：Normal PDF ---
x_norm = np.linspace(-4, 4, 300)
pdf_norm = stats.norm.pdf(x_norm, 0, 1)
axes[1, 0].plot(x_norm, pdf_norm, color='coral', linewidth=2)
axes[1, 0].fill_between(x_norm, pdf_norm, alpha=0.3, color='coral')
axes[1, 0].set_title('連續：Standard Normal PDF', fontsize=13)
axes[1, 0].set_xlabel('x')
axes[1, 0].set_ylabel('f(x)')

# --- 連續：Normal CDF ---
cdf_norm = stats.norm.cdf(x_norm, 0, 1)
axes[1, 1].plot(x_norm, cdf_norm, color='coral', linewidth=2)
axes[1, 1].set_title('連續：Standard Normal CDF', fontsize=13)
axes[1, 1].set_xlabel('x')
axes[1, 1].set_ylabel('F(x) = P(X <= x)')

plt.tight_layout()
plt.show()

**觀察重點：**
- PMF 使用長條圖 (bar)，每根代表一個確切機率值
- PDF 使用平滑曲線，機率由曲線下方面積表示
- CDF 是單調遞增函數，從 0 趨近到 1

---
## Part B -- 離散分佈 (Discrete Distributions)

### B1. Bernoulli 分佈

最簡單的離散分佈：單次試驗，結果只有「成功 (1)」或「失敗 (0)」。

$$P(X = 1) = p, \quad P(X = 0) = 1 - p$$

- 期望值 $E[X] = p$
- 變異數 $Var(X) = p(1-p)$

In [ ]:
# Bernoulli 分佈：以 Titanic 存活率為例
p_survive = df['Survived'].mean()
print(f"Titanic 存活率 p = {p_survive:.4f}")

# 模擬 Bernoulli 試驗
bernoulli_rv = stats.bernoulli(p=p_survive)
samples = bernoulli_rv.rvs(size=1000)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar([0, 1], [bernoulli_rv.pmf(0), bernoulli_rv.pmf(1)],
       color=['salmon', 'seagreen'], edgecolor='white', width=0.5)
ax.set_xticks([0, 1])
ax.set_xticklabels(['死亡 (0)', '存活 (1)'])
ax.set_ylabel('機率')
ax.set_title(f'Bernoulli 分佈 (p = {p_survive:.3f})', fontsize=14)
for i, v in enumerate([bernoulli_rv.pmf(0), bernoulli_rv.pmf(1)]):
    ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=12)
plt.show()

### B2. 二項分佈 (Binomial Distribution)

進行 $n$ 次獨立的 Bernoulli 試驗，成功次數 $X$ 服從二項分佈：

$$P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}, \quad k = 0, 1, \ldots, n$$

- 期望值 $E[X] = np$
- 變異數 $Var(X) = np(1-p)$

In [ ]:
# 應用情境：10 位乘客中有幾位存活？
n_passengers = 10
binom_rv = stats.binom(n=n_passengers, p=p_survive)

k_values = np.arange(0, n_passengers + 1)
pmf_values = binom_rv.pmf(k_values)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PMF
colors = ['seagreen' if k >= 5 else 'salmon' for k in k_values]
axes[0].bar(k_values, pmf_values, color=colors, edgecolor='white')
axes[0].set_xlabel('存活人數 k')
axes[0].set_ylabel('P(X = k)')
axes[0].set_title(f'Binomial PMF (n={n_passengers}, p={p_survive:.3f})', fontsize=13)
axes[0].set_xticks(k_values)

# CDF
cdf_values = binom_rv.cdf(k_values)
axes[1].step(k_values, cdf_values, where='mid', color='steelblue', linewidth=2)
axes[1].scatter(k_values, cdf_values, color='steelblue', zorder=5)
axes[1].set_xlabel('存活人數 k')
axes[1].set_ylabel('P(X <= k)')
axes[1].set_title(f'Binomial CDF (n={n_passengers}, p={p_survive:.3f})', fontsize=13)
axes[1].set_xticks(k_values)

plt.tight_layout()
plt.show()

In [ ]:
# 計算具體機率
print("=== 10 位乘客中的存活機率 ===")
print(f"恰好 5 人存活：P(X=5) = {binom_rv.pmf(5):.4f}")
print(f"至多 3 人存活：P(X<=3) = {binom_rv.cdf(3):.4f}")
print(f"至少 5 人存活：P(X>=5) = {1 - binom_rv.cdf(4):.4f}")
print(f"\n期望值 E[X] = {binom_rv.mean():.2f}")
print(f"標準差 SD[X] = {binom_rv.std():.2f}")

### B3. Poisson 分佈

描述在固定時間或空間內，事件發生次數的分佈。參數 $\mu$（lambda）代表平均發生次數。

$$P(X = k) = \frac{e^{-\mu} \mu^k}{k!}, \quad k = 0, 1, 2, \ldots$$

- 期望值 $E[X] = \mu$
- 變異數 $Var(X) = \mu$
- 特性：期望值等於變異數- 當 $n$ 很大、$p$ 很小時，Binomial 可近似為 Poisson（$\mu = np$）

In [ ]:
# Binomial 到 Poisson 的近似
n_large, p_small = 1000, 0.005
mu_approx = n_large * p_small  # mu = np = 5

k_vals = np.arange(0, 15)
binom_pmf = stats.binom.pmf(k_vals, n_large, p_small)
poisson_pmf = stats.poisson.pmf(k_vals, mu_approx)

fig, ax = plt.subplots(figsize=(10, 5))
width = 0.35
ax.bar(k_vals - width/2, binom_pmf, width, label=f'Binomial(n={n_large}, p={p_small})', color='steelblue')
ax.bar(k_vals + width/2, poisson_pmf, width, label=f'Poisson(mu={mu_approx})', color='coral')
ax.set_xlabel('k')
ax.set_ylabel('P(X = k)')
ax.set_title('Binomial vs Poisson 近似 (n 大, p 小時)', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 應用情境：每天平均 5 筆退貨，某天退 8 筆的機率？
mu = 5
poisson_rv = stats.poisson(mu=mu)

k_range = np.arange(0, 16)
pmf_poisson = poisson_rv.pmf(k_range)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PMF
bar_colors = ['orangered' if k == 8 else 'steelblue' for k in k_range]
axes[0].bar(k_range, pmf_poisson, color=bar_colors, edgecolor='white')
axes[0].set_xlabel('退貨筆數 k')
axes[0].set_ylabel('P(X = k)')
axes[0].set_title(f'Poisson PMF (mu={mu})', fontsize=13)
axes[0].annotate(f'P(X=8)={poisson_rv.pmf(8):.4f}',
                 xy=(8, poisson_rv.pmf(8)),
                 xytext=(10, poisson_rv.pmf(8) + 0.05),
                 arrowprops=dict(arrowstyle='->', color='red'),
                 fontsize=11, color='red')

# 比較不同 mu
for mu_val in [2, 5, 10]:
    k_ext = np.arange(0, 20)
    axes[1].plot(k_ext, stats.poisson.pmf(k_ext, mu=mu_val),
                marker='o', markersize=4, label=f'mu={mu_val}')
axes[1].set_xlabel('k')
axes[1].set_ylabel('P(X = k)')
axes[1].set_title('Poisson PMF：不同 mu 值比較', fontsize=13)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# 計算具體機率
print(f"每天平均 {mu} 筆退貨：")
print(f"  某天恰好退 8 筆的機率：P(X=8) = {poisson_rv.pmf(8):.4f}")
print(f"  某天退貨超過 8 筆的機率：P(X>8) = {1 - poisson_rv.cdf(8):.4f}")
print(f"  某天退貨不超過 3 筆的機率：P(X<=3) = {poisson_rv.cdf(3):.4f}")

---
## Part C -- 連續分佈 (Continuous Distributions)

### C1. 均勻分佈 (Uniform Distribution)

在區間 $[a, b]$ 內每個值出現的機率密度相同：

$$f(x) = \frac{1}{b - a}, \quad a \leq x \leq b$$

- 期望值 $E[X] = \frac{a+b}{2}$
- 變異數 $Var(X) = \frac{(b-a)^2}{12}$

In [ ]:
# Uniform Distribution
a, b = 0, 10
x_uni = np.linspace(-1, 11, 300)
uniform_rv = stats.uniform(loc=a, scale=b - a)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x_uni, uniform_rv.pdf(x_uni), color='teal', linewidth=2)
axes[0].fill_between(x_uni, uniform_rv.pdf(x_uni), alpha=0.3, color='teal')
axes[0].set_title(f'Uniform PDF  [a={a}, b={b}]', fontsize=13)
axes[0].set_xlabel('x')
axes[0].set_ylabel('f(x)')

axes[1].plot(x_uni, uniform_rv.cdf(x_uni), color='teal', linewidth=2)
axes[1].set_title(f'Uniform CDF  [a={a}, b={b}]', fontsize=13)
axes[1].set_xlabel('x')
axes[1].set_ylabel('F(x)')

plt.tight_layout()
plt.show()

### C2. 常態分佈 (Normal Distribution)

統計學中最重要的連續分佈，又稱高斯分佈 (Gaussian Distribution)。

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} e^{-\frac{(x-\mu)^2}{2\sigma^2}}$$

- 由兩個參數完全決定：平均數 $\mu$（位置）和標準差 $\sigma$（寬度）
- 對稱於 $\mu$，鐘型曲線
- 標準常態分佈：$\mu = 0, \sigma = 1$

#### 68-95-99.7 法則 (Empirical Rule)

| 範圍 | 涵蓋機率 |
|------|----------|
| $\mu \pm 1\sigma$ | 約 68.27% |
| $\mu \pm 2\sigma$ | 約 95.45% |
| $\mu \pm 3\sigma$ | 約 99.73% |

In [ ]:
# 常態分佈 PDF 與 68-95-99.7 法則
mu_n, sigma_n = 0, 1
x = np.linspace(-4, 4, 500)
pdf_vals = stats.norm.pdf(x, mu_n, sigma_n)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(x, pdf_vals, 'k-', linewidth=2)

# 3 sigma 區域 (99.7%)
x_3s = np.linspace(-3, 3, 300)
ax.fill_between(x_3s, stats.norm.pdf(x_3s), alpha=0.15, color='green', label='99.7% (3 sigma)')

# 2 sigma 區域 (95.45%)
x_2s = np.linspace(-2, 2, 200)
ax.fill_between(x_2s, stats.norm.pdf(x_2s), alpha=0.25, color='blue', label='95.45% (2 sigma)')

# 1 sigma 區域 (68.27%)
x_1s = np.linspace(-1, 1, 100)
ax.fill_between(x_1s, stats.norm.pdf(x_1s), alpha=0.35, color='red', label='68.27% (1 sigma)')

# 標記
for s, label in [(-3, '-3s'), (-2, '-2s'), (-1, '-1s'), (0, 'mu'),
                  (1, '+1s'), (2, '+2s'), (3, '+3s')]:
    ax.axvline(s, color='grey', linestyle='--', alpha=0.5, linewidth=0.8)
    ax.text(s, -0.02, label, ha='center', fontsize=10)

ax.set_title('Standard Normal Distribution -- 68-95-99.7 法則', fontsize=14)
ax.set_xlabel('x (z-score)')
ax.set_ylabel('f(x)')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

#### Z-score 標準化

將任何常態分佈轉換為標準常態分佈：

$$z = \frac{x - \mu}{\sigma}$$

- $z$ 值表示「距離平均數幾個標準差」
- 轉換後可以使用標準常態表查機率

In [ ]:
# Z-score 標準化範例
age_clean = df['Age'].dropna()
mu_age = age_clean.mean()
sigma_age = age_clean.std()

z_scores = (age_clean - mu_age) / sigma_age

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(age_clean, bins=30, density=True, color='steelblue', edgecolor='white', alpha=0.7)
axes[0].axvline(mu_age, color='red', linestyle='--', label=f'mu = {mu_age:.1f}')
axes[0].set_title('Titanic Age -- 原始分佈', fontsize=13)
axes[0].set_xlabel('Age')
axes[0].legend()

axes[1].hist(z_scores, bins=30, density=True, color='coral', edgecolor='white', alpha=0.7)
axes[1].axvline(0, color='red', linestyle='--', label='mu = 0')
axes[1].set_title('Titanic Age -- Z-score 標準化後', fontsize=13)
axes[1].set_xlabel('z')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"標準化前：mean={age_clean.mean():.2f}, std={age_clean.std():.2f}")
print(f"標準化後：mean={z_scores.mean():.4f}, std={z_scores.std():.4f}")

In [ ]:
# 常態分佈機率計算範例
# 假設 Age ~ N(29.7, 14.5^2)，求 20~40 歲之間的機率
p_20_40 = stats.norm.cdf(40, loc=mu_age, scale=sigma_age) - stats.norm.cdf(20, loc=mu_age, scale=sigma_age)
print(f"P(20 <= Age <= 40) = {p_20_40:.4f}")

# 求 Age 的第 90 百分位數
age_p90 = stats.norm.ppf(0.90, loc=mu_age, scale=sigma_age)
print(f"Age 第 90 百分位數 = {age_p90:.1f} 歲")

# 求超過 50 歲的機率
p_over_50 = 1 - stats.norm.cdf(50, loc=mu_age, scale=sigma_age)
print(f"P(Age > 50) = {p_over_50:.4f}")

In [ ]:
# 不同 mu 和 sigma 對常態分佈的影響
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x_range = np.linspace(-10, 15, 500)

# 不同 mu
for mu_val in [-2, 0, 3, 5]:
    axes[0].plot(x_range, stats.norm.pdf(x_range, mu_val, 1),
                label=f'mu={mu_val}, sigma=1', linewidth=2)
axes[0].set_title('不同 mu 值（位置移動）', fontsize=13)
axes[0].legend()
axes[0].set_xlabel('x')

# 不同 sigma
for sigma_val in [0.5, 1, 2, 3]:
    axes[1].plot(x_range, stats.norm.pdf(x_range, 0, sigma_val),
                label=f'mu=0, sigma={sigma_val}', linewidth=2)
axes[1].set_title('不同 sigma 值（寬度變化）', fontsize=13)
axes[1].legend()
axes[1].set_xlabel('x')

plt.tight_layout()
plt.show()

### C3. t 分佈 (Student's t-Distribution)

當樣本數較小（通常 $n < 30$）且母體標準差未知時，使用 t 分佈取代常態分佈。

- 參數：自由度 $df = n - 1$
- $df$ 愈大，t 分佈愈接近標準常態分佈
- 尾巴比常態分佈更厚 (heavier tails)，反映小樣本的額外不確定性

In [ ]:
# 比較不同自由度的 t 分佈 vs 標準常態分佈
x = np.linspace(-5, 5, 500)

fig, ax = plt.subplots(figsize=(10, 6))

# 標準常態
ax.plot(x, stats.norm.pdf(x), 'k-', linewidth=2.5, label='Normal (0, 1)')

# 不同 df 的 t 分佈
df_values = [1, 3, 10, 30]
colors = ['red', 'orange', 'green', 'blue']
for df_val, color in zip(df_values, colors):
    ax.plot(x, stats.t.pdf(x, df=df_val), color=color,
            linewidth=1.5, linestyle='--', label=f't (df={df_val})')

ax.set_title('t 分佈 vs 常態分佈：不同自由度 (df) 比較', fontsize=14)
ax.set_xlabel('x')
ax.set_ylabel('f(x)')
ax.legend()
plt.tight_layout()
plt.show()

**觀察重點：**
- df=1 的 t 分佈尾巴最厚，峰值最低
- df=30 時已非常接近常態分佈
- 這就是為什麼小樣本假設檢定要使用 t 分佈 -- 它更保守

### C4. 卡方分佈 (Chi-squared Distribution)

- 記作 $\chi^2(k)$，其中 $k$ 為自由度
- 定義：$k$ 個獨立標準常態隨機變數的平方和
- 只取非負值，右偏分佈
- 應用：適合度檢定 (Goodness-of-fit)、獨立性檢定、變異數檢定

In [ ]:
# Chi-squared 與 F 分佈簡介
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chi-squared
x_chi = np.linspace(0, 25, 500)
for k in [1, 3, 5, 10]:
    axes[0].plot(x_chi, stats.chi2.pdf(x_chi, df=k),
                linewidth=2, label=f'df={k}')
axes[0].set_title('Chi-squared 分佈', fontsize=13)
axes[0].set_xlabel('x')
axes[0].set_ylabel('f(x)')
axes[0].legend()
axes[0].set_ylim(0, 0.5)

# F Distribution
x_f = np.linspace(0.01, 5, 500)
for d1, d2 in [(5, 5), (10, 10), (5, 20), (20, 20)]:
    axes[1].plot(x_f, stats.f.pdf(x_f, dfn=d1, dfd=d2),
                linewidth=2, label=f'df1={d1}, df2={d2}')
axes[1].set_title('F 分佈', fontsize=13)
axes[1].set_xlabel('x')
axes[1].set_ylabel('f(x)')
axes[1].legend()
axes[1].set_ylim(0, 1.0)

plt.tight_layout()
plt.show()

### C5. F 分佈

- 兩個獨立卡方分佈各除以自由度後的比值
- 參數：分子自由度 $d_1$、分母自由度 $d_2$
- 只取非負值
- 應用：ANOVA（變異數分析）、迴歸分析中的 F 檢定

---
## Part D -- QQ Plot 常態性判斷

### D1. QQ Plot 原理

**Q-Q Plot (Quantile-Quantile Plot)** 將資料的實際分位數與理論分佈（通常為常態分佈）的分位數進行比較：

- 若資料服從常態分佈 --> 點會落在 45 度對角線上
- 若資料右偏 --> 右端的點會向上彎曲
- 若資料左偏 --> 左端的點會向下彎曲
- 若資料有厚尾 --> 兩端都偏離對角線

In [ ]:
# QQ Plot 比較：Age vs Fare
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Age -- 接近常態
stats.probplot(df['Age'].dropna(), plot=axes[0])
axes[0].set_title('QQ Plot: Age（接近常態）', fontsize=13)
axes[0].get_lines()[0].set_color('steelblue')
axes[0].get_lines()[0].set_markersize(4)

# Fare -- 明顯偏態
stats.probplot(df['Fare'].dropna(), plot=axes[1])
axes[1].set_title('QQ Plot: Fare（明顯右偏）', fontsize=13)
axes[1].get_lines()[0].set_color('coral')
axes[1].get_lines()[0].set_markersize(4)

plt.tight_layout()
plt.show()

In [ ]:
# 搭配直方圖更直觀
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Age'].dropna(), bins=30, density=True,
             color='steelblue', edgecolor='white', alpha=0.7)
x_fit = np.linspace(df['Age'].min(), df['Age'].max(), 200)
axes[0].plot(x_fit, stats.norm.pdf(x_fit, mu_age, sigma_age),
             'r-', linewidth=2, label='Normal fit')
axes[0].set_title('Age 分佈 vs 常態曲線', fontsize=13)
axes[0].legend()

fare_clean = df['Fare'].dropna()
axes[1].hist(fare_clean, bins=50, density=True,
             color='coral', edgecolor='white', alpha=0.7)
x_fit_f = np.linspace(fare_clean.min(), fare_clean.max(), 200)
axes[1].plot(x_fit_f, stats.norm.pdf(x_fit_f, fare_clean.mean(), fare_clean.std()),
             'r-', linewidth=2, label='Normal fit')
axes[1].set_title('Fare 分佈 vs 常態曲線', fontsize=13)
axes[1].legend()

plt.tight_layout()
plt.show()

**觀察：**
- Age 的分佈大致呈鐘型，QQ Plot 中的點大致沿對角線 --> 接近常態
- Fare 的分佈明顯右偏，QQ Plot 中右端點大幅偏離對角線 --> 不服從常態分佈

### D2. Shapiro-Wilk 常態性統計檢定

除了 QQ Plot 的目視判斷，也可以用統計檢定量化：
- **Shapiro-Wilk test**：H0 = 資料服從常態分佈
- p-value < 0.05 則拒絕 H0，即資料不服從常態

In [ ]:
# Shapiro-Wilk 常態性檢定
from scipy.stats import shapiro

# Age
stat_age, p_age = shapiro(df['Age'].dropna().sample(n=200, random_state=42))
print(f"Age:  W-statistic = {stat_age:.4f}, p-value = {p_age:.4f}")
print(f"  --> {'不拒絕 H0 (可能服從常態)' if p_age >= 0.05 else '拒絕 H0 (不服從常態)'}")

# Fare
stat_fare, p_fare = shapiro(df['Fare'].dropna().sample(n=200, random_state=42))
print(f"\nFare: W-statistic = {stat_fare:.4f}, p-value = {p_fare:.4f}")
print(f"  --> {'不拒絕 H0 (可能服從常態)' if p_fare >= 0.05 else '拒絕 H0 (不服從常態)'}")

---
## Part E -- 實務應用：分佈辨識與轉換

### E1. 辨識 Fare 的分佈特徵

許多統計方法（如 t 檢定、ANOVA）假設資料服從常態分佈。當資料明顯偏態時，常用的處理方式是進行**對數轉換 (Log Transformation)**。

In [ ]:
# Fare 的基本統計量
print("=== Fare 描述統計 ===")
print(fare_clean.describe())
print(f"\n偏態係數 (Skewness)：{fare_clean.skew():.4f}")
print(f"峰態係數 (Kurtosis)：{fare_clean.kurtosis():.4f}")
print("\n偏態係數 > 1 表示嚴重右偏，需要考慮轉換")

In [ ]:
# 對數轉換：log1p(x) = log(1 + x)，避免 log(0) 的問題
fare_log = np.log1p(fare_clean)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 轉換前
axes[0, 0].hist(fare_clean, bins=50, density=True,
                color='coral', edgecolor='white', alpha=0.7)
axes[0, 0].set_title('Fare -- 原始分佈', fontsize=13)
axes[0, 0].set_xlabel('Fare')

stats.probplot(fare_clean, plot=axes[0, 1])
axes[0, 1].set_title('Fare -- QQ Plot（轉換前）', fontsize=13)

# 轉換後
axes[1, 0].hist(fare_log, bins=50, density=True,
                color='seagreen', edgecolor='white', alpha=0.7)
x_fit_log = np.linspace(fare_log.min(), fare_log.max(), 200)
axes[1, 0].plot(x_fit_log,
                stats.norm.pdf(x_fit_log, fare_log.mean(), fare_log.std()),
                'r-', linewidth=2, label='Normal fit')
axes[1, 0].set_title('log1p(Fare) -- 對數轉換後', fontsize=13)
axes[1, 0].set_xlabel('log(1 + Fare)')
axes[1, 0].legend()

stats.probplot(fare_log, plot=axes[1, 1])
axes[1, 1].set_title('log1p(Fare) -- QQ Plot（轉換後）', fontsize=13)

plt.tight_layout()
plt.show()

print(f"轉換前偏態：{fare_clean.skew():.4f}")
print(f"轉換後偏態：{fare_log.skew():.4f}")

**結論：**
- 對數轉換後，Fare 的分佈明顯更接近常態
- QQ Plot 中的點更貼近對角線
- 偏態係數大幅下降
- 在後續的假設檢定或建模中，使用 `log1p(Fare)` 會比原始 Fare 更合適

---
## 重點整理：常見分佈速查表

| 分佈 | 類型 | 參數 | scipy.stats | 典型應用 |
|------|------|------|-------------|----------|
| **Bernoulli** | 離散 | $p$ | `bernoulli(p)` | 二元結果（是/否） |
| **Binomial** | 離散 | $n, p$ | `binom(n, p)` | n 次試驗的成功次數 |
| **Poisson** | 離散 | $\mu$ | `poisson(mu)` | 單位時間內事件次數 |
| **Uniform** | 連續 | $a, b$ | `uniform(a, b-a)` | 等機率分佈 |
| **Normal** | 連續 | $\mu, \sigma$ | `norm(mu, sigma)` | 自然現象、中央極限定理 |
| **t** | 連續 | $df$ | `t(df)` | 小樣本假設檢定 |
| **Chi-squared** | 連續 | $k$ | `chi2(df)` | 適合度檢定、獨立性檢定 |
| **F** | 連續 | $d_1, d_2$ | `f(dfn, dfd)` | ANOVA、迴歸 F 檢定 |

---
## 練習題

### 基礎題

1. 某產品的不良率為 8%，隨機抽取 20 件，求恰好有 2 件不良品的機率。使用 `stats.binom` 計算。

2. 某客服中心每小時平均接到 12 通電話，求某小時接到超過 15 通的機率。使用 `stats.poisson` 計算。

3. 某班級學生身高服從 $N(170, 5^2)$，計算身高在 165 到 175 之間的機率。使用 `stats.norm` 的 CDF。

### 進階題

4. 使用 Titanic 資料集的 `Age` 欄位，繪製直方圖並疊加常態分佈曲線。計算該欄位的偏態係數 (skewness)，判斷是否接近常態。

5. 比較 `SibSp`（兄弟姐妹/配偶數）的實際分佈與 Poisson 分佈的擬合程度。提示：先計算 `SibSp` 的平均值作為 Poisson 的 mu 參數。

### 挑戰題

6. 對 Titanic 的所有數值欄位（Age, Fare, SibSp, Parch）繪製 QQ Plot（2x2 子圖），並為每個欄位判斷最可能的分佈類型。對明顯偏態的欄位嘗試 `np.log1p` 轉換並比較轉換前後的 QQ Plot。

---

<- [Ch03 -- 機率基礎與條件機率](03_機率基礎與條件機率.ipynb) | [Ch05 ->](05_抽樣分佈與中央極限定理.ipynb)